In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import boto3
import pickle

try:
    import optbinning
except:
    ! pip install optbinning
    
try:
    import catboost
except:
    ! pip install catboost

#### Functions

In [ ]:
# download from s3
def download_from_s3(str_local_path, str_bucket_path, str_project):
    # init client
    cls_client = boto3.client(
        's3',
    )
    # download file
    cls_client.download_file(
        str_project, 
        str_bucket_path, 
        str_local_path,
    )

#### Constants

In [ ]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

str_dirname_output = './output'

#### Output dir

In [ ]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Import data

In [ ]:
str_filename = 'df.gzip'
str_uri = f's3://20241112-simple-model-test/08_prep_data/{str_filename}'
df = pd.read_parquet(
    str_uri,
)
# drop
list_cols = [col for col in df.columns if 'tu_pmthx' in col]
list_cols.append('applicationdayofweek__app')
df.drop(list_cols, axis=1, inplace=True)
# sort
df.sort_values(by='request_datetime', ascending=True, inplace=True)
df

#### Import preprocessor

In [ ]:
list_str_filename = [
    'preprocessing.py',
    'cls_model_preprocessing.pkl',
]
for str_filename in list_str_filename:
    str_bucket_path = f'01_ad/02_model/noPTImodel10/00_preprocessing/01_create_preprocessor/{str_filename}'
    str_local_path = f'./{str_filename}'
    download_from_s3(
        str_local_path=str_local_path,
        str_bucket_path=str_bucket_path,
        str_project='20231010-gen-xii',
    )
cls_model_preprocessing = pickle.load(open(str_local_path, 'rb'))

#### Preprocess data

In [ ]:
df = cls_model_preprocessing.transform(df)
# rm
list_str_filename = [
    'preprocessing.py',
    'cls_model_preprocessing.pkl',
]
for str_filename in tqdm(list_str_filename):
    os.remove(str_filename)
# show
df

#### AD predictions

In [ ]:
str_filename = 'final_model.pkl'
str_model = '01_ad'
str_bucket_path = f'{str_model}/02_model/noPTImodel10/03_final_model/{str_filename}'
str_local_path = f'./{str_filename}'
download_from_s3(
    str_local_path=str_local_path,
    str_bucket_path=str_bucket_path,
    str_project='20231010-gen-xii',
)
cls_model_inference = pickle.load(open(str_local_path, 'rb'))['model_inference']
os.remove(str_local_path)
# predict
list_cols_model = list(cls_model_inference.feature_names_)
df['ad'] = cls_model_inference.predict_proba(df[list_cols_model])[:,1]
# show
df

#### PD predictions

In [ ]:
str_filename = 'final_model.pkl'
str_model = '02_pricing_pd'
str_bucket_path = f'{str_model}/02_model/noPTImodel10/03_final_model/{str_filename}'
str_local_path = f'./{str_filename}'
download_from_s3(
    str_local_path=str_local_path,
    str_bucket_path=str_bucket_path,
    str_project='20231010-gen-xii',
)
cls_model_inference = pickle.load(open(str_local_path, 'rb'))['model_inference']
os.remove(str_local_path)
# predict
list_cols_model = list(cls_model_inference.feature_names_)
df['pd'] = cls_model_inference.predict_proba(df[list_cols_model])[:,1]
# show
df

#### LGD predictions

In [ ]:
str_filename = 'final_model.pkl'
str_model = '03_pricing_lgd'
str_bucket_path = f'{str_model}/02_model/noPTImodel10/03_final_model/{str_filename}'
str_local_path = f'./{str_filename}'
download_from_s3(
    str_local_path=str_local_path,
    str_bucket_path=str_bucket_path,
    str_project='20231010-gen-xii',
)
cls_model_inference = pickle.load(open(str_local_path, 'rb'))['model_inference']
os.remove(str_local_path)
# predict
list_cols_model = list(cls_model_inference.feature_names_)
df['lgd'] = cls_model_inference.predict(df[list_cols_model])
# show
df

#### Convert non-numeric to string

In [ ]:
for col in tqdm(df.columns):
    str_dtype = df[col].dtype
    if str_dtype not in ['int64','float64']:
        df[col] = df[col].astype(str)
    else:
        pass

#### Write to s3

In [ ]:
%%time

str_filename = 'df_clean_w_pred.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_filename}'
df.to_parquet(
    str_uri,
    compression='gzip',
)